# A Recommender System for MovieLens-1M

We build and evaluate a movie recommender on the MovieLens-1M dataset
(1,000,209 ratings from 6,040 users across 3,706 movies). The central
challenge is *sparsity*: users have rated only a tiny fraction of the
available movies, and our task is to predict which unseen movies a user
would enjoy from this incomplete signal.

This notebook covers the foundation of the project: we explore the data,
establish a rigorous evaluation methodology built around time-based
splitting and top-*k* ranking metrics, and implement a popularity baseline
against which all later models are measured. Matrix factorization and a
neural approach follow in subsequent notebooks.

In [67]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.metrics import precision_at_k, recall_at_k, ndcg_at_k, evaluate, safe_mean
from src.baseline import popular_movies, recommend_popular

## The Data

The MovieLens-1M dataset consists of three files; we focus on `ratings.dat`,
which records user-movie ratings. Each row is a single *rating event* — one
user assigning one rating to one movie at one point in time — with four
fields: `user_id`, `movie_id`, `rating` (an integer from 1 to 5), and a Unix
`timestamp`. Note that ratings are not aggregated: quantities like a movie's
average rating or its number of ratings are *computed* from these raw events
rather than stored directly.

In [84]:
ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    names=["user_id", "movie_id", "rating", "timestamp"],
    engine="python",
)
ratings.head()

,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


## Sparsity: the core challenge

We can think of the data as a large matrix with users as rows and movies as
columns, where each cell holds a rating. Most cells are empty — the vast
majority of users have rated only a small fraction of the available movies.
We quantify this with the *density*: the fraction of all possible user-movie
pairs that actually have a rating.

$$\text{density} = \frac{\text{number of ratings}}{\text{number of users} \times \text{number of movies}}$$

These empty cells are not missing data to be cleaned away — they *are* the
prediction problem. Recommending a movie to a user is precisely the act of
estimating the value of an empty cell.

In [86]:
n_users = ratings["user_id"].nunique()
n_movies = ratings["movie_id"].nunique()
n_ratings = len(ratings)

density = n_ratings / (n_users * n_movies)
print(f"Users: {n_users}, Movies: {n_movies}, Ratings: {n_ratings}")
print(f"Density: {density:.4f} ({density:.2%} of all possible ratings)")

Users: 6040, Movies: 3706, Ratings: 1000209
Density: 0.0447 (4.47% of all possible ratings)


We convert the Unix timestamp into a human-readable datetime, which we will
need for the time-based train/test split described below.

In [85]:
ratings["datetime"] = pd.to_datetime(ratings["timestamp"], unit="s")
ratings_sorted = ratings.sort_values("timestamp").reset_index(drop=True)
cutoff = int(len(ratings_sorted) * 0.8)
train = ratings_sorted[:cutoff]
test = ratings_sorted[cutoff:]

### Why split by time?

We sort all ratings chronologically and assign the earliest 80% to training
and the latest 20% to testing. A random split would leak future information:
some of a user's later ratings would land in training while earlier ones fall
into test, letting the model implicitly "see the future." Since a recommender
is deployed to predict *future* behavior from *past* behavior, a time-based
split mirrors real use and gives an honest estimate of performance.

In [79]:
print("Train:", train["datetime"].min(), "to", train["datetime"].max())
print("Test: ", test["datetime"].min(), "to", test["datetime"].max())

Train: 2000-04-25 23:05:32 to 2000-12-02 14:52:18
Test:  2000-12-02 14:52:18 to 2003-02-28 17:49:50


### Cold-start users

A time-based split has a side effect worth examining: some users appear only
in the test period and never in training. A model that learns from
interaction history has nothing to go on for these users — this is the
*user cold-start* problem. Before evaluating, we measure how many test users
are cold.

In [80]:
train_users = set(train["user_id"])
test_users = set(test["user_id"])
cold_users = test_users - train_users
print(f"Cold users (in test, not train): {len(cold_users)} of {len(test_users)}")

Cold users (in test, not train): 640 of 1783


### Why so many cold users?

This reflects how people rate movies: a user typically rates a batch of
films over a short period and then goes quiet for a long time. Because
ratings are clustered per-user in time, slicing the data chronologically
inevitably places a wave of "new arrivals" in the test period — users whose
entire history falls after the cutoff.

We do not attempt to solve cold-start in this project. In practice it can be
addressed with onboarding questionnaires, demographic side-features, or a
popularity-based fallback, all of which require signal beyond the interaction
matrix. Instead, we restrict our core evaluation to *warm* users (those
present in training) and note the exclusion as a limitation.

In [82]:
test_warm = test[test["user_id"].isin(train_users)]
print(f"Warm test interactions: {len(test_warm)} (from {test_warm['user_id'].nunique()} users)")

Warm test interactions: 104540 (from 1143 users)


## Evaluation Metrics

A natural first instinct is to evaluate a recommender by how accurately it
predicts ratings — e.g. with RMSE between predicted and actual scores. But
this measures the wrong thing. In practice a recommender produces a *ranked
list*, and the user only ever sees the top few items. RMSE weights every
prediction equally and ignores order entirely, so a model can have excellent
RMSE while ranking the user's favorite movie tenth.

We therefore evaluate with **top-*k* ranking metrics**, which score only the
top *k* recommendations and care about their ordering: Precision@*k*,
Recall@*k*, and NDCG@*k*.

### Precision@*k*

Of the *k* items we recommend, what fraction are actually relevant to the
user? We define an item as *relevant* if the user rated it 4 or higher, on
the grounds that a recommendation should surface movies the user genuinely
liked, not merely tolerated.

$$\text{Precision@}k = \frac{|\{\text{recommended top-}k\} \cap \{\text{relevant}\}|}{k}$$

Precision is always defined, since the denominator *k* is fixed.

### Recall@*k*

Of *all* the items the user found relevant, what fraction did we manage to
place in the top *k*? The numerator is the same as for precision — relevant
items among our top *k* — but we now divide by the total number of relevant
items.

$$\text{Recall@}k = \frac{|\{\text{recommended top-}k\} \cap \{\text{relevant}\}|}{|\{\text{relevant}\}|}$$

Unlike precision, recall is *undefined* when a user has no relevant items
(the denominator is zero). We exclude such users from the averaged recall
rather than scoring them zero, which would unfairly understate performance.
Note also that Recall@*k* is bounded above when a user has more than *k*
relevant items, since at most *k* can fit in the list.

### NDCG@*k*

Precision and recall treat the top *k* as an unordered set: a relevant item
counts the same whether it sits at position 1 or position *k*. But order
matters — a recommender that ranks the user's favorite first is better than
one that buries it. NDCG captures this.

We build it in three steps. First, **Discounted Cumulative Gain** sums the
relevance of recommended items, discounting each by its position so that
items lower in the list contribute less:

$$\text{DCG@}k = \sum_{i=1}^{k} \frac{\text{rel}_i}{\log_2(i + 1)}$$

where $\text{rel}_i = 1$ if the item at position $i$ is relevant and $0$
otherwise. Second, **Ideal DCG** is the DCG of the best possible ranking —
all relevant items placed at the very top:

$$\text{IDCG@}k = \sum_{i=1}^{\min(k,\, R)} \frac{1}{\log_2(i + 1)}$$

where $R$ is the number of relevant items. Finally, we normalize:

$$\text{NDCG@}k = \frac{\text{DCG@}k}{\text{IDCG@}k}$$

Dividing by the ideal places every user on a $[0, 1]$ scale, where $1.0$
means a perfect ranking, making scores comparable across users regardless of
how many relevant items each has.

## A Popularity Baseline

Before building anything sophisticated, we establish a baseline: recommend
the most *popular* movies to everyone, ignoring personalization entirely. We
define popularity by rating count — the number of times a movie was rated in
the training set — rather than average rating, which would let a single
5-star rating vault an obscure film to the top.

For each user we recommend the most popular movies they have not already
rated in training. This baseline is deliberately simple, but it is not a
strawman: popular movies are popular precisely because many people like them,
so the baseline is often surprisingly hard to beat. It sets the bar that any
personalized model must clear to justify its complexity.

In [87]:
warm_users = list(test_warm["user_id"].unique())
recs = recommend_popular(train, warm_users, k=10)

relevant_ratings = test_warm[test_warm["rating"] >= 4]
relevant_by_user = relevant_ratings.groupby("user_id")["movie_id"].apply(set).to_dict()

In [88]:
results = evaluate(recs, relevant_by_user, k=10)
print(f"Precision@10: {results['precision']:.4f}")
print(f"Recall@10:    {results['recall']:.4f}")
print(f"NDCG@10:      {results['ndcg']:.4f}")

Precision@10: 0.1959
Recall@10:    0.0493
NDCG@10:      0.2145


## Results

| Metric | Score |
|---|---|
| Precision@10 | 0.1959 |
| Recall@10 | 0.0493 |
| NDCG@10 | 0.2145 |

The baseline performs respectably, which is itself instructive. **Precision@10
≈ 0.20** means about two of every ten recommended movies were relevant — high
for a non-personalized model, but not surprising: popular movies are popular
*because* many users like them, so recommending them hits often almost by
construction.

**Recall@10 ≈ 0.05** is low primarily because of a structural ceiling: many
users have far more than ten relevant movies, and only ten can fit in the
list, so even a perfect ranking would capture a small fraction. The lack of
personalization contributes, but the *k*-cap is the dominant effect.

**NDCG@10 ≈ 0.21** is best read not as "good" or "bad" in isolation, but as
the *bar*. Any personalized model we build must clear these numbers to justify
its added complexity — and because popularity is a genuinely strong baseline,
that is not guaranteed.

## Next Steps

This notebook established the foundation: an understanding of the data and its
sparsity, a leakage-free evaluation methodology, and a popularity baseline to
beat. The natural next step is a *personalized* model.

In the next notebook we implement **matrix factorization**, which addresses
sparsity directly: rather than working with the mostly-empty user-movie
matrix, we learn a compact latent vector for each user and each movie, so that
a rating is approximated by their dot product. This compresses millions of
sparse entries into a few learned dimensions and, crucially, generalizes to
the empty cells — the predictions we actually care about. We will also
incorporate per-user and per-item bias terms to account for systematic
differences in how generously users rate and how highly movies are rated.